# 03. B2B Commercial Activation, Propensity Scoring & Field Route Sequencing

Executive-grade B2B outbound intelligence pipeline transforming resolved entities from Stage 02 into commercial revenue:
1. **Stage-02 Production Contract Handoff:** Ingests deduplicated canonical corporate accounts and actionable physical sites with bidirectional English/Spanish schema compatibility.
2. **4-Digit SCIAN Chronotype Imputer:** Pinpoints 7:00 AM – 10:00 AM morning coffee demand windows across manufacturing shifts, medical handovers, and corporate offices.
3. **Digital Maturity & Ethical DuckDuckGo OSINT Rescue:** Audits contact channels and enriches high-priority gap accounts lacking verified email addresses.
4. **Propensity Scoring & Unit Economics EMV:** Fuses CRITIC/AdaBoost 5D scores, morning chronotype alignment, and campus payroll mass into Expected Monthly B2B Coffee Spend (MXN).
5. **Activation Channel Routing & Offer Playbooks:** Generates 4-tier tailored coffee offerings and sequences physical field visits from *The Coffee — Paseo MS* via 2-Opt TSP.
6. **Strict Clean-Room Dual Export:** Full private CRM/SDR campaign packages alongside a zero-PII public sector aggregate index.
7. **Interactive Folium Activation Map:** Executive map with 1 km / 3 km operational rings, route polylines, and LinkedIn Sales Navigator shortcuts.

In [1]:
# Cell 3.1: Load Stage-02 Production Contracts & Self-Healing JSON Contracts
import json
from pathlib import Path
import polars as pl
from IPython.display import HTML, display

def find_file(rel_path: str) -> Path:
    candidates = [Path(rel_path), Path(f"../{rel_path}"), Path(f"notebooks/{rel_path}")]
    for c in candidates:
        if c.exists(): return c
    return Path(rel_path)

# Bidirectional English/Spanish Schema Mapper
def normalize_stage3_schema(df: pl.DataFrame) -> pl.DataFrame:
    mapping = {
        "codigo_act": "scian_code",
        "per_ocu": "employee_strata",
        "correoelec": "email",
        "www": "website",
        "telefono": "phone",
        "nombre_estab": "business_name",
        "raz_social": "legal_name",
        "mun": "municipality",
    }
    rename_dict = {k: v for k, v in mapping.items() if k in df.columns and v not in df.columns}
    return df.rename(rename_dict) if rename_dict else df

# Ingest Stage 02 Datasets
entities_path = find_file("data/processed/entities_resolved.parquet")
canon_path = find_file("data/processed/tables/stage2_canonical_action_master.parquet")
if not canon_path.exists(): canon_path = find_file("data/processed/tables/stage2_canonical_action_master.csv")
if not canon_path.exists(): canon_path = find_file("data/processed/stage2_canonical_action_master.parquet")

site_path = find_file("data/processed/tables/stage2_site_action_list.parquet")
if not site_path.exists(): site_path = find_file("data/processed/tables/stage2_site_action_list.csv")
if not site_path.exists(): site_path = find_file("data/processed/stage2_site_action_list.parquet")

gap_path = find_file("data/processed/tables/stage2_missing_enrichment_gap.csv")

df_entities = normalize_stage3_schema(pl.read_parquet(entities_path))
df_canonical = pl.read_parquet(canon_path) if str(canon_path).endswith(".parquet") else pl.read_csv(canon_path)
df_sites = pl.read_parquet(site_path) if str(site_path).endswith(".parquet") else pl.read_csv(site_path)
df_gap = pl.read_csv(gap_path) if gap_path.exists() else pl.DataFrame()

# Ingest or Self-Heal Data Contracts
chrono_path = find_file("data_contracts/scian_chronotype_mapping.json")
if chrono_path.exists():
    with open(chrono_path, "r", encoding="utf-8") as f: chronotype_mapping = json.load(f)
else:
    chronotype_mapping = {
        "31": {"chronotype": "Early_Bird_6AM", "description": "Manufactura"},
        "32": {"chronotype": "Early_Bird_6AM", "description": "Manufactura"},
        "33": {"chronotype": "Early_Bird_6AM", "description": "Manufactura / Automotriz"},
        "48": {"chronotype": "Early_Bird_6AM", "description": "Transporte y Logistica"},
        "49": {"chronotype": "Early_Bird_6AM", "description": "Almacenamiento"},
        "52": {"chronotype": "Office_Standard_8AM", "description": "Servicios Financieros"},
        "54": {"chronotype": "Office_Standard_8AM", "description": "Servicios Profesionales"},
        "55": {"chronotype": "Office_Standard_8AM", "description": "Corporativos"},
        "62": {"chronotype": "Early_Bird_6AM", "description": "Salud"},
        "72": {"chronotype": "Night_Shift_10PM", "description": "Hospedaje y Alimentos"},
    }
    chrono_path.parent.mkdir(parents=True, exist_ok=True)
    with open(chrono_path, "w", encoding="utf-8") as f: json.dump(chronotype_mapping, f, indent=2)

prop_path = find_file("data_contracts/scian_b2b_propensity.json")
if prop_path.exists():
    with open(prop_path, "r", encoding="utf-8") as f: propensity_weights = json.load(f)
else:
    propensity_weights = {
        "33": {"weight": 95.0, "sector_name": "Fabricacion de Equipo de Transporte"},
        "54": {"weight": 90.0, "sector_name": "Servicios Profesionales y Cientificos"},
        "52": {"weight": 85.0, "sector_name": "Servicios Financieros y de Seguros"},
        "48": {"weight": 80.0, "sector_name": "Transporte y Logistica"},
        "72": {"weight": 75.0, "sector_name": "Hoteleria Corporativa"},
    }
    prop_path.parent.mkdir(parents=True, exist_ok=True)
    with open(prop_path, "w", encoding="utf-8") as f: json.dump(propensity_weights, f, indent=2)

# Enrich canonical table with primary site geographical attributes
primary_sites = df_sites.filter(pl.col("role_in_campus") == "primary_site")
meta_cols = ["canonical_entity_id", "scian_code", "scian_description", "b2b_sector", "distance_km", "morning_traffic_score", "lat", "lon", "street_name"]
existing_meta = [c for c in meta_cols if c in primary_sites.columns]
if existing_meta:
    df_canonical = df_canonical.join(primary_sites.select(existing_meta), on="canonical_entity_id", how="left")

print(f" Loaded Stage-02 Resolved Universe: {len(df_entities):,} establishments")
print(f" Loaded Deduplicated Canonical Master: {len(df_canonical):,} corporate accounts")
print(f" Loaded Actionable Physical Sites: {len(df_sites):,} locations")
print(f" Loaded OSINT Rescue Gap Queue: {len(df_gap):,} accounts")

# Render Tabla 1: Resumen de Handoff Stage-02 -> Stage-03
t1_payroll_mxn = df_canonical["consolidated_payroll_mxn"].sum()
t1_headcount = df_canonical["total_est_employees_midpoint"].sum()
t1_gap_count = len(df_gap)

html_t1 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">📋 Tabla 1: Resumen de Handoff Operativo (Stage 02 ➔ Stage 03)</h4>
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 12px;">
        <div style="background: white; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px; text-align: center;">
            <div style="font-size: 11px; color: #64748b; font-weight: bold;">Cuentas Canónicas ABM</div>
            <div style="font-size: 22px; font-weight: bold; color: #0f766e; margin-top: 2px;">{len(df_canonical):,}</div>
            <div style="font-size: 10px; color: #16a34a; margin-top: 2px;">100% deduplicadas</div>
        </div>
        <div style="background: white; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px; text-align: center;">
            <div style="font-size: 11px; color: #64748b; font-weight: bold;">Sitios Físicos en Ruta</div>
            <div style="font-size: 22px; font-weight: bold; color: #1d4ed8; margin-top: 2px;">{len(df_sites):,}</div>
            <div style="font-size: 10px; color: #3b82f6; margin-top: 2px;">+153.1% cobertura</div>
        </div>
        <div style="background: white; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px; text-align: center;">
            <div style="font-size: 11px; color: #64748b; font-weight: bold;">Masa Salarial Silao</div>
            <div style="font-size: 22px; font-weight: bold; color: #7c3aed; margin-top: 2px;">${t1_payroll_mxn/1e6:.1f}M MXN</div>
            <div style="font-size: 10px; color: #8b5cf6; margin-top: 2px;">{t1_headcount:,} profesionistas</div>
        </div>
        <div style="background: white; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px; text-align: center;">
            <div style="font-size: 11px; color: #64748b; font-weight: bold;">Cola de Rescate DDG</div>
            <div style="font-size: 22px; font-weight: bold; color: #b45309; margin-top: 2px;">{t1_gap_count}</div>
            <div style="font-size: 10px; color: #f59e0b; margin-top: 2px;">Prioridad OSINT</div>
        </div>
    </div>
</div>
"""
display(HTML(html_t1))

 Loaded Stage-02 Resolved Universe: 398 establishments
 Loaded Deduplicated Canonical Master: 108 corporate accounts
 Loaded Actionable Physical Sites: 124 locations
 Loaded OSINT Rescue Gap Queue: 26 accounts


In [2]:
# Cell 3.2: 4-Digit SCIAN Chronotype & 7:00-10:00 AM Shift-Change Imputer
# MIGRATES TO: src/mx_osint/enrichment/chronotype_imputer.py::impute_chronotype

def impute_morning_chronotype_4digit(scian_code: str | None, academic_level: str | None = None) -> dict:
    code = str(scian_code or "").strip()
    prefix2 = code[:2]
    prefix3 = code[:3]
    prefix4 = code[:4]

    if prefix4 in ["3361", "3362", "3363", "5413"] or prefix2 in ["31", "32", "33", "23"] or prefix3 in ["484", "493"]:
        return {
            "chronotype_key": "Turno_Ingenieria_630_730AM",
            "time_window": "06:30 - 08:00 AM",
            "demand_driver": "Cambio de Turno Operativo & Juntas de Piso",
            "product_fit": "Coffee Box Térmico 10-20 pax + Desayuno de Ingeniería",
            "badge": "<span style='background:#fef3c7; color:#92400e; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>🏭 TURNO PLANTA (6:30-8:00 AM)</span>"
        }
    elif prefix3 in ["621", "622", "623"] or prefix2 == "62":
        return {
            "chronotype_key": "Guardia_Medica_700_830AM",
            "time_window": "07:00 - 08:30 AM",
            "demand_driver": "Relevo de Guardia Médica & Visita Clínica",
            "product_fit": "Pack Café Especialidad Baristas + Servicio Express",
            "badge": "<span style='background:#fee2e2; color:#991b1b; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>🏥 GUARDIA MÉDICA (7:00-8:30 AM)</span>"
        }
    elif prefix4 in ["5415", "5221", "7211"] or prefix2 in ["52", "55"] or academic_level == "corporate_professional":
        return {
            "chronotype_key": "Ejecutivo_Oficina_800_930AM",
            "time_window": "08:00 - 09:30 AM",
            "demand_driver": "Llegada Directiva & Salas de Consejo",
            "product_fit": "Barra Ejecutiva & Convenio Corporativo CFDI 4.0",
            "badge": "<span style='background:#dbeafe; color:#1e40af; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>💼 OFICINA / EJECUTIVO (8:00-9:30 AM)</span>"
        }
    else:
        return {
            "chronotype_key": "Apertura_Comercial_900_1000AM",
            "time_window": "09:00 - 10:00 AM",
            "demand_driver": "Arranque Comercial & Pausa Matutina",
            "product_fit": "Take-away & Entrega Programada",
            "badge": "<span style='background:#f1f5f9; color:#475569; padding:2px 8px; border-radius:10px; font-weight:bold; font-size:10px;'>🏪 COMERCIO (9:00-10:00 AM)</span>"
        }

chrono_results = [
    impute_morning_chronotype_4digit(r.get("scian_code"), r.get("workforce_academic_level"))
    for r in df_canonical.iter_rows(named=True)
]

df_canonical = df_canonical.with_columns([
    pl.Series("chronotype_key", [c["chronotype_key"] for c in chrono_results]),
    pl.Series("morning_time_window", [c["time_window"] for c in chrono_results]),
    pl.Series("demand_driver", [c["demand_driver"] for c in chrono_results]),
    pl.Series("product_fit", [c["product_fit"] for c in chrono_results]),
    pl.Series("chronotype_badge", [c["badge"] for c in chrono_results]),
])

# Render Tabla 2: Matriz de Cronotipos Matutinos
chrono_rollup = (
    df_canonical.group_by(["chronotype_badge", "morning_time_window", "demand_driver", "product_fit"])
    .agg([
        pl.len().alias("cuentas"),
        pl.col("site_count").sum().alias("sitios_totales"),
        pl.col("total_est_employees_midpoint").sum().alias("headcount"),
        pl.col("consolidated_payroll_mxn").sum().alias("nomina_total"),
    ])
    .sort("nomina_total", descending=True)
)

t2_rows = []
for r in chrono_rollup.iter_rows(named=True):
    t2_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 8px 10px;">{r['chronotype_badge']}</td>
        <td style="padding: 8px 10px; font-weight: bold; color: #0f766e; text-align: center;">{r['morning_time_window']}</td>
        <td style="padding: 8px 10px; text-align: center; font-weight: bold;">{r['cuentas']} ({r['sitios_totales']} sitios)</td>
        <td style="padding: 8px 10px; text-align: right; font-weight: 600;">{r['headcount']:,} pers.</td>
        <td style="padding: 8px 10px; text-align: right; font-weight: bold; color: #1e3a8a;">${r['nomina_total']/1e6:.1f}M MXN</td>
        <td style="padding: 8px 10px; font-size: 11px; color: #334155;">{r['product_fit']}</td>
    </tr>
    """)

html_t2 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">⏰ Tabla 2: Matriz de Cronotipos Matutinos (7:00–10:00 AM) por Masa Salarial y Oferta</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #1e293b; color: white;">
            <tr>
                <th style="padding: 8px 10px;">Perfil de Cronotipo</th>
                <th style="padding: 8px 10px; text-align: center;">Ventana Horaria</th>
                <th style="padding: 8px 10px; text-align: center;">Cuentas Canónicas</th>
                <th style="padding: 8px 10px; text-align: right;">Empleados</th>
                <th style="padding: 8px 10px; text-align: right;">Masa Salarial / Mes</th>
                <th style="padding: 8px 10px;">Solución de Café B2B Recomendada</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t2_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Segmentación basada en el código SCIAN de 4 dígitos y nivel de especialización laboral de INEGI/IMSS.</i></p>
</div>
"""
display(HTML(html_t2))

Perfil de Cronotipo,Ventana Horaria,Cuentas Canónicas,Empleados,Masa Salarial / Mes,Solución de Café B2B Recomendada
🏭 TURNO PLANTA (6:30-8:00 AM),06:30 - 08:00 AM,36 (42 sitios),"8,645 pers.",$219.4M MXN,Coffee Box Térmico 10-20 pax + Desayuno de Ingeniería
💼 OFICINA / EJECUTIVO (8:00-9:30 AM),08:00 - 09:30 AM,33 (38 sitios),"2,115 pers.",$44.2M MXN,Barra Ejecutiva & Convenio Corporativo CFDI 4.0
🏪 COMERCIO (9:00-10:00 AM),09:00 - 10:00 AM,30 (34 sitios),"1,765 pers.",$30.5M MXN,Take-away & Entrega Programada
🏥 GUARDIA MÉDICA (7:00-8:30 AM),07:00 - 08:30 AM,9 (10 sitios),315 pers.,$6.4M MXN,Pack Café Especialidad Baristas + Servicio Express


In [3]:
# Cell 3.3: Digital Maturity Audit & Ethical DuckDuckGo OSINT Rescue
# MIGRATES TO: src/mx_osint/enrichment/digital_maturity.py & src/mx_osint/activation/ddg_enricher.py
import time
from loguru import logger

def classify_digital_maturity_record(row: dict) -> dict:
    has_em = bool(row.get("effective_contact") and str(row.get("effective_contact")).strip() not in ["", "-", "None"])
    conf = (row.get("email_confidence") or "none").lower()
    has_web = bool(row.get("website") and str(row.get("website")).strip() not in ["", "-", "None"])
    has_ph = bool(row.get("phone") and str(row.get("phone")).strip() not in ["", "-", "None"])

    # Score 0 to 3
    score = 0
    if has_em and conf in ["high", "medium-high"]: score += 2
    elif has_em: score += 1
    if has_web: score += 1

    return {
        "has_verified_email": bool(has_em and conf in ["high", "medium-high"]),
        "has_website": has_web,
        "has_phone": has_ph,
        "digital_maturity_score": min(3, score),
        "digital_gap_flag": not has_em or conf in ["low", "none"],
    }

maturity_results = [classify_digital_maturity_record(r) for r in df_canonical.iter_rows(named=True)]
df_canonical = df_canonical.with_columns([
    pl.Series("has_verified_email", [m["has_verified_email"] for m in maturity_results]),
    pl.Series("has_website", [m["has_website"] for m in maturity_results]),
    pl.Series("has_phone", [m["has_phone"] for m in maturity_results]),
    pl.Series("digital_maturity_score", [m["digital_maturity_score"] for m in maturity_results]),
    pl.Series("digital_gap_flag", [m["digital_gap_flag"] for m in maturity_results]),
])

# Ethical DuckDuckGo Discovery Helper with local JSON cache
ddg_cache_path = find_file("data/processed/ddg_osint_cache.json")
ddg_cache = {}
if ddg_cache_path.exists():
    try:
        with open(ddg_cache_path, "r", encoding="utf-8") as f: ddg_cache = json.load(f)
    except Exception: ddg_cache = {}

def discover_channel_ddg(entity_name: str, state: str = "Silao Guanajuato", max_results: int = 2) -> dict:
    cache_key = f"{entity_name}::{state}"
    if cache_key in ddg_cache: return ddg_cache[cache_key]

    query = f'"{entity_name}" "{state}" contacto OR telefono OR recursos humanos'
    out = {"status": "searched", "top_url": None, "snippet": None}
    try:
        from duckduckgo_search import DDGS
        time.sleep(0.5)
        with DDGS() as ddgs:
            res = list(ddgs.text(query, max_results=max_results))
            if res:
                out = {"status": "found", "top_url": res[0].get("href"), "snippet": res[0].get("body", "")[:120]}
    except Exception as e:
        out = {"status": "skipped_or_offline", "top_url": f"https://www.google.com/search?q={entity_name}+Silao", "snippet": "Búsqueda web directa"}

    ddg_cache[cache_key] = out
    return out

# Execute DDG on top gap candidates
gap_candidates = df_canonical.filter(pl.col("digital_gap_flag") == True).sort("max_b2b_5d_score", descending=True).head(15)
ddg_results = []
for r in gap_candidates.iter_rows(named=True):
    name = r["canonical_parent_name"]
    res = discover_channel_ddg(name)
    ddg_results.append({
        "canonical_entity_id": r["canonical_entity_id"],
        "canonical_parent_name": name,
        "max_b2b_5d_score": r["max_b2b_5d_score"],
        "current_channel": r["best_contact_channel"],
        "ddg_status": res.get("status"),
        "ddg_url": res.get("top_url") or "-",
        "ddg_snippet": res.get("snippet") or "Sin snippet",
    })

ddg_cache_path.parent.mkdir(parents=True, exist_ok=True)
with open(ddg_cache_path, "w", encoding="utf-8") as f:
    json.dump(ddg_cache, f, indent=2, ensure_ascii=False)

# Render Tabla 3: Auditoría y Rescate OSINT
t3_rows = []
for rank, d in enumerate(ddg_results, start=1):
    link_tag = f"<a href='{d['ddg_url']}' target='_blank' style='color:#0a66c2; text-decoration:none; font-weight:bold;'>🔗 Ver Enlace</a>" if d['ddg_url'] != "-" else "-"
    t3_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #64748b;">{rank}</td>
        <td style="padding: 6px 10px; font-weight: 600; color: #1e3a8a;">{d['canonical_parent_name']}</td>
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #0284c7;">{float(d['max_b2b_5d_score']):.2f}</td>
        <td style="padding: 6px 8px; font-size: 11px; color: #b45309;">{d['current_channel']}</td>
        <td style="padding: 6px 8px; font-size: 11px; color: #334155;">{d['ddg_snippet']}</td>
        <td style="padding: 6px 8px; text-align: center; font-size: 11px;">{link_tag}</td>
    </tr>
    """)

html_t3 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🔍 Tabla 3: Auditoría de Madurez Digital y Rescate OSINT (Top 15 Cuentas Gap)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #0f766e; color: white;">
            <tr>
                <th style="padding: 8px 8px; text-align: center;">#</th>
                <th style="padding: 8px 10px;">Cuenta Matriz</th>
                <th style="padding: 8px 8px; text-align: center;">Score 5D</th>
                <th style="padding: 8px 8px;">Canal Inicial</th>
                <th style="padding: 8px 8px;">Hallazgo OSINT (DuckDuckGo)</th>
                <th style="padding: 8px 8px; text-align: center;">Evidencia Web</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t3_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Rescate ético rate-limited y cacheado en data/processed/ddg_osint_cache.json.</i></p>
</div>
"""
display(HTML(html_t3))

#,Cuenta Matriz,Score 5D,Canal Inicial,Hallazgo OSINT (DuckDuckGo),Evidencia Web
1,BANCO AZTECA INSTITUCION DE BANCA MULTIPLE,6.03,Field Sales + LinkedIn ABM,Sin snippet,-
2,CENTRO DE SERVICIOS E INGENIERIA,5.89,Field Sales + LinkedIn ABM,Sin snippet,-
3,BANCO DEL BAJIO INSTITUCION DE BANCA MULTIPLE,5.88,Field Sales + LinkedIn ABM,Sin snippet,-
4,BBVA BANCOMER INSTITUCION DE BANCA MULTIPLE GRUPO FINANCIERO BBVA,5.73,Field Sales + LinkedIn ABM,Sin snippet,-
5,BANCO NACIONAL DE MEXICO INTEGRANTE DEL GRUPO FINANCIERO BANAMEX,5.65,Website / Synthesized Domain Email,"Con la Banca por Internet de BBVA podrás realizar cualquier consulta, transferencia y pago de forma más segura y en cues",🔗 Ver Enlace
6,RIGGING AND CONSTRUCTION SERVICES,5.40,Field Sales + LinkedIn ABM,Sin snippet,-
7,RED NACIONAL ULTIMA MILLA,5.15,Field Sales + LinkedIn ABM,Sin snippet,-
8,DESPACHO CONTABLE IBARRA ROCHA Y ORTEGA,5.13,Direct Phone + LinkedIn ABM,Sin snippet,-
9,L&V5 MEXICO,5.09,Field Sales + LinkedIn ABM,Directorio Telefónico de México El directorio digital telefónico más grande de México. Registre completamente GRATIS su,🔗 Ver Enlace
10,SIMSA SISTEMAS DE NEUMATICOS DEL BAJIO,4.85,Field Sales + LinkedIn ABM,Sin snippet,-


In [4]:
# Cell 3.4: B2B Propensity Scoring (0-100) & Expected Monthly Coffee Spend (EMV en MXN)
# MIGRATES TO: src/mx_osint/enrichment/b2b_propensity.py::score_b2b_propensity

def calculate_b2b_emv_record(r: dict) -> dict:
    score_5d = float(r.get("max_b2b_5d_score") or 5.0)
    payroll = float(r.get("consolidated_payroll_mxn") or 500000.0)
    headcount = int(r.get("total_est_employees_midpoint") or 25)
    dist = float(r.get("distance_km") or 1.5)
    chrono = r.get("chronotype_key", "Office_Standard_8AM")
    conf = r.get("email_confidence", "none")

    # Chronotype base capture rates
    chrono_rates = {
        "Turno_Ingenieria_630_730AM": 0.28,
        "Guardia_Medica_700_830AM": 0.32,
        "Ejecutivo_Oficina_800_930AM": 0.35,
        "Apertura_Comercial_900_1000AM": 0.18,
    }
    base_rate = chrono_rates.get(chrono, 0.20)
    dist_decay = max(0.40, 1.0 - (dist / 8.0))
    capture_rate = round(min(0.40, base_rate * dist_decay), 3)

    # White collar headcount estimate
    white_collar = max(1, round(headcount * 0.40)) if headcount > 50 else max(1, headcount)
    avg_ticket_mxn = 65.0
    working_days = 20
    emv_mxn = round(white_collar * capture_rate * avg_ticket_mxn * working_days, 2)

    # 4-Pillar Propensity Score 0 to 100
    # Pillar 1: 5D Multi-Criteria Model (50 pts max)
    p1 = min(50.0, score_5d * 5.0)
    # Pillar 2: Chronotype & Peak Exposure (20 pts max)
    p2 = 20.0 if "630" in chrono or "800" in chrono else 12.0
    # Pillar 3: Payroll Purchasing Power (15 pts max)
    p3 = min(15.0, (payroll / 5000000.0) * 15.0)
    # Pillar 4: Contact Confidence & Deliverability (15 pts max)
    p4 = 15.0 if conf == "high" else (10.0 if conf in ["medium-high", "medium"] else 5.0)

    total_score = round(min(100.0, max(15.0, p1 + p2 + p3 + p4)), 1)

    return {
        "b2b_propensity_score": total_score,
        "capture_rate": capture_rate,
        "white_collar_headcount": white_collar,
        "est_monthly_coffee_rev_mxn": emv_mxn,
    }

emv_results = [calculate_b2b_emv_record(r) for r in df_canonical.iter_rows(named=True)]
df_canonical = df_canonical.with_columns([
    pl.Series("b2b_propensity_score", [e["b2b_propensity_score"] for e in emv_results]),
    pl.Series("capture_rate", [e["capture_rate"] for e in emv_results]),
    pl.Series("white_collar_headcount", [e["white_collar_headcount"] for e in emv_results]),
    pl.Series("est_monthly_coffee_rev_mxn", [e["est_monthly_coffee_rev_mxn"] for e in emv_results]),
]).sort("b2b_propensity_score", descending=True)

# Render Tabla 4: Top 20 Cuentas por Propensión B2B y Valor Estimado EMV
top20 = df_canonical.head(20)

t4_rows = []
for rank, r in enumerate(top20.iter_rows(named=True), start=1):
    emv_fmt = f"${r['est_monthly_coffee_rev_mxn']:,.0f} MXN"
    payroll_fmt = f"${r['consolidated_payroll_mxn']/1e6:.2f}M"
    t4_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 8px; font-weight: bold; text-align: center; color: #64748b;">{rank}</td>
        <td style="padding: 6px 10px; font-weight: 600; color: #1e3a8a;">{r['canonical_parent_name']}</td>
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #0284c7;">{r['b2b_propensity_score']} pts</td>
        <td style="padding: 6px 8px; text-align: right; font-weight: bold; color: #0f766e;">{emv_fmt}</td>
        <td style="padding: 6px 8px; text-align: right; color: #475569;">{r['total_est_employees_midpoint']:,} pers.</td>
        <td style="padding: 6px 8px; text-align: right; color: #1e3a8a; font-weight: 600;">{payroll_fmt}</td>
        <td style="padding: 6px 8px; font-size: 11px;">{r['morning_time_window']}</td>
        <td style="padding: 6px 8px; font-size: 11px; color:#15803d; font-weight:600;">{r['best_contact_channel']}</td>
    </tr>
    """)

html_t4 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">💎 Tabla 4: Top 20 Cuentas Canónicas por Propensión B2B (0–100) y Valor Mensual Estimado (EMV)</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #1e1b4b; color: white;">
            <tr>
                <th style="padding: 8px 8px; text-align: center;">#</th>
                <th style="padding: 8px 10px;">Cuenta Corporativa Matriz</th>
                <th style="padding: 8px 8px; text-align: center;">Propensión B2B</th>
                <th style="padding: 8px 8px; text-align: right;">Gasto Mensual Estimado (EMV)</th>
                <th style="padding: 8px 8px; text-align: right;">Plantilla Total</th>
                <th style="padding: 8px 8px; text-align: right;">Masa Salarial</th>
                <th style="padding: 8px 8px;">Ventana Matutina</th>
                <th style="padding: 8px 8px;">Canal Asignado</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t4_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Modelo unit-economics: Headcount × Tasa Captura × Ticket Promedio ($65 MXN) × 20 días laborales.</i></p>
</div>
"""
display(HTML(html_t4))

#,Cuenta Corporativa Matriz,Propensión B2B,Gasto Mensual Estimado (EMV),Plantilla Total,Masa Salarial,Ventana Matutina,Canal Asignado
1,CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTES,90.2 pts,"$68,796 MXN",700 pers.,$21.01M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
2,MAHLE BEHR RIO BRAVO,89.0 pts,"$38,584 MXN",350 pers.,$10.51M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
3,JOHNAN DE MEXICO,89.0 pts,"$18,109 MXN",175 pers.,$5.25M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
4,AUMOVIO MEXICO,89.0 pts,"$126,126 MXN","1,225 pers.",$33.49M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
5,GENERAL MOTORS DE MEXICO,88.8 pts,"$33,852 MXN",350 pers.,$13.52M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
6,ANDROID DE MEXICO,88.0 pts,"$19,019 MXN",175 pers.,$5.25M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
7,P&S INGENIERIA ELECTROMECANICAS,87.7 pts,"$21,395 MXN",195 pers.,$5.71M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
8,AAM MAQUILADORA MEXICO,87.0 pts,"$26,390 MXN",250 pers.,$6.94M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
9,L&V 5 MEXICO,86.5 pts,"$39,494 MXN",350 pers.,$10.51M,06:30 - 08:00 AM,Direct Email (Verified DENUE)
10,ARNESES ELECTRICOS AUTOMOTRICES,85.6 pts,"$35,854 MXN",350 pers.,$10.51M,06:30 - 08:00 AM,Direct Email (Verified DENUE)


In [5]:
# Cell 3.5: Activation Channel Router, Offer Playbook & Field-Visit Route Sequencer
# MIGRATES TO: src/mx_osint/activation/channel_classifier.py::classify_channel
import math

PASEO_MS_LAT, PASEO_MS_LON = 20.953792, -101.416053

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2.0) ** 2
    return 2.0 * r * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

def assign_offer_playbook(r: dict) -> dict:
    hc = int(r.get("total_est_employees_midpoint") or 10)
    scian = str(r.get("scian_code", ""))[:2]
    dist = float(r.get("distance_km") or 1.5)
    name = r.get("canonical_parent_name") or "la empresa"

    if (scian in ["31", "32", "33", "23"] or hc >= 100):
        code = "Playbook A"
        name_offer = "Convenio Corporativo & Coffee Box Juntas Matutinas (7:00–9:00 AM) con CFDI 4.0"
        pitch = f"Solución integral de abastecimiento continuo de café de especialidad para líneas de ingeniería y coffee breaks ejecutivos en juntas directivas de {name}."
    elif dist <= 2.0 and scian in ["52", "62", "54"]:
        code = "Playbook B"
        name_offer = "Membresía Vecino Corporativo Paseo MS (15% Off 7–10 AM + Delivery Express)"
        pitch = f"Convenio de descuento corporativo del 15% para el personal de {name} y entregas ágiles a pie de sucursal durante la mañana."
    elif scian == "72":
        code = "Playbook C"
        name_offer = "Alianza Huésped Ejecutivo / Barra de Café Especialidad para Eventos"
        pitch = f"Alianza para abastecer coffee breaks en salones ejecutivos y paquetes de bienvenida para huéspedes VIP en {name}."
    else:
        code = "Playbook D"
        name_offer = "Servicio Ágil de Café de Especialidad para Oficinas y Despachos"
        pitch = f"Suministro programado de café de grano y bebidas preparadas para la jornada matutina del equipo de {name}."

    return {"playbook_code": code, "playbook_name": name_offer, "playbook_pitch": pitch}

playbook_results = [assign_offer_playbook(r) for r in df_canonical.iter_rows(named=True)]
df_canonical = df_canonical.with_columns([
    pl.Series("b2b_offer_playbook_code", [p["playbook_code"] for p in playbook_results]),
    pl.Series("b2b_offer_playbook", [p["playbook_name"] for p in playbook_results]),
    pl.Series("b2b_offer_pitch", [p["playbook_pitch"] for p in playbook_results]),
])

# Route assignment for physical sites
def assign_route_corridor(lat: float, lon: float) -> str:
    d = haversine_km(PASEO_MS_LAT, PASEO_MS_LON, lat, lon)
    if d <= 1.0:
        return "Ruta 1: Eje Paseo MS / Joya / Hospital General (<=1 km)"
    elif d <= 2.2:
        return "Ruta 2: Corredor Financiero Silao Centro (1-2 km)"
    else:
        return "Ruta 3: Cluster Industrial Norte / FIPASI / Colinas (>2 km)"

route_names = [assign_route_corridor(float(r["lat"]), float(r["lon"])) for r in df_sites.iter_rows(named=True)]
df_sites = df_sites.with_columns(pl.Series("field_sales_route", route_names))

# Solve TSP per route
def solve_route_tsp(subset_df: pl.DataFrame) -> pl.DataFrame:
    recs = subset_df.to_dicts()
    n = len(recs)
    if n == 0: return subset_df

    unvisited = list(range(n))
    curr_lat, curr_lon = PASEO_MS_LAT, PASEO_MS_LON
    order = []
    while unvisited:
        nxt = min(unvisited, key=lambda idx: haversine_km(curr_lat, curr_lon, float(recs[idx]["lat"]), float(recs[idx]["lon"])))
        order.append(nxt)
        curr_lat, curr_lon = float(recs[nxt]["lat"]), float(recs[nxt]["lon"])
        unvisited.remove(nxt)

    # 2-Opt swap
    def d_tot(od):
        td = haversine_km(PASEO_MS_LAT, PASEO_MS_LON, float(recs[od[0]]["lat"]), float(recs[od[0]]["lon"]))
        for i in range(len(od) - 1):
            td += haversine_km(float(recs[od[i]]["lat"]), float(recs[od[i]]["lon"]), float(recs[od[i+1]]["lat"]), float(recs[od[i+1]]["lon"]))
        return td

    improved = True
    while improved:
        improved = False
        for i in range(len(order) - 1):
            for j in range(i + 1, len(order)):
                candidate = order[:i] + order[i:j+1][::-1] + order[j+1:]
                if d_tot(candidate) < d_tot(order) - 1e-4:
                    order = candidate
                    improved = True

    ordered_out = []
    c_lat, c_lon = PASEO_MS_LAT, PASEO_MS_LON
    cum_d = 0.0
    for seq, idx in enumerate(order, start=1):
        item = dict(recs[idx])
        dest_lat, dest_lon = float(item["lat"]), float(item["lon"])
        leg = haversine_km(c_lat, c_lon, dest_lat, dest_lon)
        cum_d += leg
        item["route_stop_order"] = seq
        item["leg_km"] = round(leg, 2)
        item["cumulative_km"] = round(cum_d, 2)
        item["est_drive_min"] = round(leg * 2.5)
        ordered_out.append(item)
        c_lat, c_lon = dest_lat, dest_lon

    return pl.DataFrame(ordered_out)

r1 = solve_route_tsp(df_sites.filter(pl.col("field_sales_route").str.contains("Ruta 1")))
r2 = solve_route_tsp(df_sites.filter(pl.col("field_sales_route").str.contains("Ruta 2")))
r3 = solve_route_tsp(df_sites.filter(pl.col("field_sales_route").str.contains("Ruta 3")))
df_field_routed = pl.concat([r1, r2, r3])

# Render Tabla 5: Plan Maestro de Activación Comercial
top_routes = df_field_routed.sort("b2b_5d_score", descending=True).head(15)

t5_rows = []
for rank, r in enumerate(top_routes.iter_rows(named=True), start=1):
    role_tag = "<span style='background:#dbeafe; color:#1e40af; padding:1px 6px; border-radius:4px; font-weight:bold; font-size:10px;'>Principal</span>" if r.get("role_in_campus") == "primary_site" else "<span style='background:#f1f5f9; color:#475569; padding:1px 6px; border-radius:4px; font-size:10px;'>Sucursal</span>"
    t5_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #64748b;">#{r['route_stop_order']}</td>
        <td style="padding: 6px 10px; font-weight: 600; color: #1e3a8a;">{r['business_name']}<br><span style="font-size:10px; color:#64748b;">Matriz: {r['canonical_parent_name']}</span></td>
        <td style="padding: 6px 8px; text-align: center;">{role_tag}</td>
        <td style="padding: 6px 8px; font-size: 11px; font-weight:bold; color:#0f766e;">{r['field_sales_route'].split(':')[0]}</td>
        <td style="padding: 6px 8px; text-align: center; font-weight: bold; color: #0284c7;">{float(r.get('b2b_5d_score') or 0.0):.2f}</td>
        <td style="padding: 6px 8px; font-size: 11px; color: #334155;">{r.get('street_name', 'Silao')} #{r.get('street_number_ext', 'S/N')}</td>
        <td style="padding: 6px 8px; text-align: right; font-weight: bold;">{r['leg_km']} km</td>
        <td style="padding: 6px 8px; text-align: right; color:#64748b;">{r['cumulative_km']} km ({r['est_drive_min']} min)</td>
    </tr>
    """)

html_t5 = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🗺️ Tabla 5: Plan Maestro de Activación Comercial: Canales y Rutas de Campo Secuenciadas</h4>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 11px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden;">
        <thead style="background: #0f766e; color: white;">
            <tr>
                <th style="padding: 8px 8px; text-align: center;">Parada</th>
                <th style="padding: 8px 10px;">Establecimiento Físico</th>
                <th style="padding: 8px 8px; text-align: center;">Rol Sitio</th>
                <th style="padding: 8px 8px;">Ruta Asignada</th>
                <th style="padding: 8px 8px; text-align: center;">Score 5D</th>
                <th style="padding: 8px 8px;">Ubicación / Calle</th>
                <th style="padding: 8px 8px; text-align: right;">Tramo</th>
                <th style="padding: 8px 8px; text-align: right;">Acumulado</th>
            </tr>
        </thead>
        <tbody>
            {''.join(t5_rows)}
        </tbody>
    </table>
    <p style="font-size: 11px; color: #64748b; margin-top: 5px;"><i>Nota: Secuencia calculada con TSP 2-Opt anclada en The Coffee Paseo MS (20.953792, -101.416053).</i></p>
</div>
"""
display(HTML(html_t5))

Parada,Establecimiento Físico,Rol Sitio,Ruta Asignada,Score 5D,Ubicación / Calle,Tramo,Acumulado
#17,BANORTE SILAOMatriz: BANCO MERCANTIL DEL NORTE INSTITUCION DE BANCA MULTIPLE GRUPO FINANCIERO BANORTE,Principal,Ruta 1,8.26,LUIS H. DUCOING #144,0.11 km,3.56 km (0 min)
#22,SERVICIO DE CALIBRACION CAPYMETMatriz: CAPYMET,Principal,Ruta 1,8.22,ACANTILADO #17,0.05 km,3.93 km (0 min)
#18,CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTESMatriz: CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTES,Principal,Ruta 3,8.04,PARAÍSO #449,0.06 km,7.06 km (0 min)
#27,BANCO COMPARTAMOS SILAOMatriz: BANCO COMPARTAMOS INSTITUCION DE BANCA MULTIPLE,Principal,Ruta 2,7.98,ÁLVARO OBREGÓN NORTE #138,0.03 km,7.91 km (0 min)
#17,LEAR SILAOMatriz: CONSORCIO INDUSTRIAL MEXICANO DE AUTOPARTES,Sucursal,Ruta 3,7.97,PARAÍSO (FEDERAL 57 PIEDRAS NEGRAS) #449,0.22 km,7.01 km (1 min)
#18,CAC SILAOMatriz: RADIO MOVIL DIPSA,Principal,Ruta 1,7.81,LUIS H. DUCOING #144,0.0 km,3.57 km (0 min)
#7,MAHLE SILAOMatriz: MAHLE BEHR RIO BRAVO,Principal,Ruta 2,7.81,FRESNO #223,0.29 km,2.68 km (1 min)
#9,AUMOVIO MEXICO MATRIZMatriz: AUMOVIO MEXICO,Principal,Ruta 3,7.79,PASEO DE LAS COLINAS #219,0.06 km,5.98 km (0 min)
#12,JOHNANMatriz: JOHNAN DE MEXICO,Principal,Ruta 3,7.79,EUCALIPTO #240,0.08 km,6.11 km (0 min)
#41,GENERAL MOTORS DE MEXICOMatriz: GENERAL MOTORS DE MEXICO,Principal,Ruta 3,7.76,GUANAJUATO SILAO #38,1.18 km,22.83 km (3 min)


In [6]:
# Cell 3.6: Strict Clean-Room Dual Export & CRM Ready Files
# MIGRATES TO: src/mx_osint/reporting/prospect_report.py::export_dual

for base_dir in [Path("data"), Path("../data"), Path("notebooks/data")]:
    if base_dir.exists():
        priv_dir = base_dir / "private"
        proc_dir = base_dir / "processed"
        priv_dir.mkdir(parents=True, exist_ok=True)
        proc_dir.mkdir(parents=True, exist_ok=True)

        # 1. Private Top-30 B2B Prospects
        top30_export = df_canonical.head(30)
        top30_export.write_csv(priv_dir / "top_30_b2b_prospects.csv")

        # 2. Private Verified Low-Risk Email Sequences
        email_ready = df_canonical.filter((pl.col("contact_risk") == "low") & (pl.col("email_confidence").is_in(["high", "medium-high"])))
        email_ready.write_csv(priv_dir / "stage3_email_sequence_ready.csv")

        # 3. Private Field Sales Routes
        df_field_routed.write_csv(priv_dir / "stage3_field_sales_routes.csv")

        # 4. Public Clean-Room Aggregate Index (Zero PII)
        pub_agg = (
            df_canonical
            .group_by(["chronotype_key", "b2b_offer_playbook_code"])
            .agg([
                pl.len().alias("total_accounts"),
                pl.col("site_count").sum().alias("total_physical_sites"),
                pl.col("total_est_employees_midpoint").sum().alias("total_headcount"),
                pl.col("consolidated_payroll_mxn").sum().alias("total_payroll_mxn"),
                pl.col("est_monthly_coffee_rev_mxn").sum().alias("total_est_monthly_coffee_rev_mxn"),
                pl.col("b2b_propensity_score").mean().alias("avg_propensity_score"),
            ])
            .sort("total_payroll_mxn", descending=True)
        )
        pub_payload = {
            "meta": {
                "generator": "mexico-commercial-osint",
                "stage": "03_b2b_propensity_activation",
                "base_anchor": "The Coffee — Paseo MS (Silao, GTO)",
                "total_canonical_accounts": len(df_canonical),
                "total_actionable_sites": len(df_field_routed),
                "total_monthly_coffee_opportunity_mxn": float(df_canonical["est_monthly_coffee_rev_mxn"].sum()),
            },
            "aggregates": pub_agg.to_dicts(),
        }
        with open(proc_dir / "public_b2b_aggregates.json", "w", encoding="utf-8") as f:
            json.dump(pub_payload, f, indent=2, ensure_ascii=False)

print(f"💾 Private Top 30 Prospects: stage3_top_30_b2b_prospects.csv ({len(top30_export)} accounts)")
print(f"💾 Private Email Ready (Low Risk): stage3_email_sequence_ready.csv ({len(email_ready)} accounts)")
print(f"💾 Private Field Sales Routes: stage3_field_sales_routes.csv ({len(df_field_routed)} sites)")
print(f"💾 Public Clean-Room Aggregate Index: public_b2b_aggregates.json (Zero PII, ${df_canonical['est_monthly_coffee_rev_mxn'].sum():,.0f} MXN/mes en demanda)")

💾 Private Top 30 Prospects: stage3_top_30_b2b_prospects.csv (30 accounts)
💾 Private Email Ready (Low Risk): stage3_email_sequence_ready.csv (85 accounts)
💾 Private Field Sales Routes: stage3_field_sales_routes.csv (124 sites)
💾 Public Clean-Room Aggregate Index: public_b2b_aggregates.json (Zero PII, $1,634,620 MXN/mes en demanda)


In [7]:
# Cell 3.7: Interactive Folium Activation & Field-Route Map
import folium
from folium.plugins import MiniMap

m_act = folium.Map(location=[PASEO_MS_LAT, PASEO_MS_LON], zoom_start=13, tiles=None)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="&copy; <a href='https://www.openstreetmap.org/copyright'>OpenStreetMap</a> contributors",
    name="🗺️ OpenStreetMap",
    control=True,
).add_to(m_act)

folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap &copy; CARTO",
    name="🏙️ CARTO Positron",
    control=True,
).add_to(m_act)

folium.LayerControl(position="topright").add_to(m_act)

# Operational Rings: 1 km walking/convenience & 3 km delivery/commute
folium.Circle(
    location=[PASEO_MS_LAT, PASEO_MS_LON],
    radius=1000,
    color="#0f766e",
    fill=True,
    fill_opacity=0.08,
    popup="<b>Anillo 1 km</b>: Vecinos Corporativos / Walk-in",
).add_to(m_act)

folium.Circle(
    location=[PASEO_MS_LAT, PASEO_MS_LON],
    radius=3000,
    color="#1d4ed8",
    fill=False,
    dash_array="5, 10",
    popup="<b>Anillo 3 km</b>: Cobertura B2B Delivery Express",
).add_to(m_act)

# Base Anchor: The Coffee Paseo MS
folium.Marker(
    location=[PASEO_MS_LAT, PASEO_MS_LON],
    popup="<b>📍 The Coffee — Paseo MS</b><br>Centro Operativo de Activación Comercial",
    tooltip="📍 The Coffee Paseo MS",
    icon=folium.Icon(color="red", icon="coffee", prefix="fa"),
).add_to(m_act)

route_colors = {
    "Ruta 1": "#16a34a",
    "Ruta 2": "#2563eb",
    "Ruta 3": "#9333ea",
}

for route_pfx, col in route_colors.items():
    r_sites = df_field_routed.filter(pl.col("field_sales_route").str.contains(route_pfx)).sort("route_stop_order")
    pts = [[PASEO_MS_LAT, PASEO_MS_LON]] + [[float(r["lat"]), float(r["lon"])] for r in r_sites.iter_rows(named=True)]
    if len(pts) > 1:
        folium.PolyLine(pts, color=col, weight=2.5, opacity=0.75, tooltip=f"Polilínea {route_pfx}").add_to(m_act)

for r in df_field_routed.iter_rows(named=True):
    r_name = r.get("field_sales_route", "")
    pfx = r_name.split(":")[0] if ":" in r_name else "Ruta"
    col = route_colors.get(pfx, "blue")

    pop_html = f"""
    <div style='font-family: Arial; font-size: 11px; min-width: 220px;'>
        <b style='color:#0f172a;'>Parada #{r['route_stop_order']}: {r['business_name']}</b><br>
        <span style='color:#64748b;'>Matriz: {r['canonical_parent_name']}</span><hr style='margin:4px 0;'>
        <b>Score 5D:</b> {float(r.get('b2b_5d_score') or 0.0):.2f} pts<br>
        <b>Ruta:</b> {pfx}<br>
        <b>Tramo:</b> {r['leg_km']} km ({r['est_drive_min']} min)<br>
        <b>Distancia Acumulada:</b> {r['cumulative_km']} km<br>
        <a href='{r.get("linkedin_search_url", "#")}' target='_blank' style='color:#0a66c2; font-weight:bold;'>LinkedIn Search &rarr;</a>
    </div>
    """
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=6 if r.get("role_in_campus") == "primary_site" else 4,
        color=col,
        fill=True,
        fill_color=col,
        fill_opacity=0.85,
        popup=folium.Popup(pop_html, max_width=300),
        tooltip=f"#{r['route_stop_order']} {r['business_name']} ({pfx})",
    ).add_to(m_act)

MiniMap(toggle_display=True).add_to(m_act)

for base_dir in [Path("data/private"), Path("../data/private"), Path("notebooks/data/private")]:
    if base_dir.exists():
        m_act.save(str(base_dir / "stage3_field_routes_map.html"))

display(m_act)

In [10]:
# Cell 3.8: Franchise-Constrained Geolocation Audit (App Pickup vs. Rappi Viability)
import numpy as np
import polars as pl
from IPython.display import HTML, display
import math

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2.0) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2.0) ** 2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# 1. Prepare Data (Fix: using 'site_count' from df_canonical schema)
sites_with_emv = df_field_routed.join(
    df_canonical.select(["canonical_entity_id", "est_monthly_coffee_rev_mxn", "site_count"]),
    on="canonical_entity_id", how="left"
).with_columns([
    (pl.col("est_monthly_coffee_rev_mxn") / pl.col("site_count").fill_null(1)).fill_null(5000.0).alias("site_emv_mxn")
])

lats = sites_with_emv["lat"].to_numpy()
lons = sites_with_emv["lon"].to_numpy()
emvs = sites_with_emv["site_emv_mxn"].to_numpy()

# 2. Define Candidate Locations
CANDIDATE_LOCATIONS = [
    {"name": "📍 Actual: The Coffee — Paseo MS", "lat": 20.953792, "lon": -101.416053, "type": "Commuter Hub"},
    {"name": "🏭 Alt 1: Parque Industrial Las Colinas", "lat": 20.968800, "lon": -101.431200, "type": "Industrial"},
    {"name": "🏦 Alt 2: Centro Financiero / Obregón", "lat": 20.954500, "lon": -101.425400, "type": "Financial"},
    {"name": "🎯 Math Centroid (EMV Gravity)", "lat": float(np.sum(lats * emvs) / np.sum(emvs)), "lon": float(np.sum(lons * emvs) / np.sum(emvs)), "type": "Theoretical"},
]

eval_rows = []
for loc in CANDIDATE_LOCATIONS:
    c_lat, c_lon = loc["lat"], loc["lon"]
    dists = np.array([haversine_km(c_lat, c_lon, float(la), float(lo)) for la, lo in zip(lats, lons)])
    
    # METRIC 1: App Pickup Potential (Steep distance decay - people won't walk >800m)
    # Formula: EMV / (1 + distance)^3  <-- Heavy penalty for distance
    app_pickup_potential = float(np.sum(emvs / np.power(1.0 + dists, 3.0)))
    
    # METRIC 2: Rappi Delivery Viability (Capped at 3.5km, flat fee penalty)
    # Only count sites within 3.5km, penalize by estimated Rappi friction
    rappi_mask = dists <= 3.5
    rappi_potential = float(np.sum(emvs[rappi_mask] * 0.70)) # 30% margin loss to Rappi
    
    # METRIC 3: Commuter Interception (Sites within 800m)
    commuter_sites = int(np.sum(dists <= 0.8)) 
    
    # Composite Franchise Score (App Pickup is King)
    # Normalizing dynamically to avoid division by zero or weird scales if EMV is small
    max_app = max([float(np.sum(emvs / np.power(1.0 + np.array([haversine_km(c["lat"], c["lon"], float(la), float(lo)) for la, lo in zip(lats, lons)]), 3.0))) for c in CANDIDATE_LOCATIONS]) or 1.0
    max_rappi = max([float(np.sum(emvs[np.array([haversine_km(c["lat"], c["lon"], float(la), float(lo)) for la, lo in zip(lats, lons)]) <= 3.5] * 0.70)) for c in CANDIDATE_LOCATIONS]) or 1.0
    
    norm_app = (app_pickup_potential / max_app) * 60
    norm_rappi = (rappi_potential / max_rappi) * 40
    franchise_score = round(norm_app + norm_rappi, 1)
    
    eval_rows.append(f"""
    <tr style="border-bottom: 1px solid #e2e8f0;">
        <td style="padding: 8px 10px; font-weight: bold; color: #0f172a;">{loc['name']}<br><small style="color:#64748b">{loc['type']}</small></td>
        <td style="padding: 8px 10px; text-align: center; font-weight: bold; color: #0284c7;">{commuter_sites} sitios</td>
        <td style="padding: 8px 10px; text-align: right; font-weight: bold; color: #16a34a;">${app_pickup_potential:,.0f} pts</td>
        <td style="padding: 8px 10px; text-align: right; color: #ea580c;">${rappi_potential:,.0f} MXN</td>
        <td style="padding: 8px 10px; text-align: center;"><span style="background:#dcfce7; color:#166534; padding:4px 10px; border-radius:12px; font-weight:bold; font-size:13px;">{franchise_score}</span></td>
    </tr>
    """)

html_geo = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 15px 0;">
    <h4 style="margin: 0 0 8px 0; color: #0f172a;">🧭 Tabla 6: Evaluación de Optimalidad para Modelo de Franquicia (App Pickup + Rappi)</h4>
    <p style="font-size: 12px; color: #475569; margin-bottom: 15px;">
        <b>Nota:</b> El "Potencial App Pickup" aplica una penalización cúbica a la distancia (los clientes no caminan >800m). 
        El "Potencial Rappi" asume una pérdida de margen del 30% y un radio máximo de 3.5km. El <b>Score Franquicia</b> pondera 60% App Pickup y 40% Rappi.
    </p>
    <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 12px; background: white; border: 1px solid #cbd5e1; border-radius: 6px; overflow: hidden; box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.1);">
        <thead style="background: #0f172a; color: white;">
            <tr>
                <th style="padding: 10px;">Ubicación Evaluada</th>
                <th style="padding: 10px; text-align: center;">Sitios "Walk-In" (&le;800m)</th>
                <th style="padding: 10px; text-align: right;">Índice App Pickup<br><small>(Mayor = Mejor)</small></th>
                <th style="padding: 10px; text-align: right;">EMV Viable por Rappi<br><small>(Después de comisiones)</small></th>
                <th style="padding: 10px; text-align: center;">Score Franquicia</th>
            </tr>
        </thead>
        <tbody>{''.join(eval_rows)}</tbody>
    </table>
</div>
"""
display(HTML(html_geo))

Ubicación Evaluada,"Sitios ""Walk-In"" (≤800m)",Índice App Pickup(Mayor = Mejor),EMV Viable por Rappi(Después de comisiones),Score Franquicia
📍 Actual: The Coffee — Paseo MSCommuter Hub,7 sitios,"$133,231 pts","$1,118,221 MXN",66.8
🏭 Alt 1: Parque Industrial Las ColinasIndustrial,22 sitios,"$298,316 pts","$960,470 MXN",94.4
🏦 Alt 2: Centro Financiero / ObregónFinancial,14 sitios,"$144,354 pts","$1,109,008 MXN",68.7
🎯 Math Centroid (EMV Gravity)Theoretical,11 sitios,"$132,920 pts","$1,109,008 MXN",66.4
